# Notebook 04 - Visualisierungen

- Tabellen & Charts (matplotlib, seaborn)
- 4-Panel Dashboard
- Geographische Karte (folium)
- Preis-Heatmap

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, matplotlib.gridspec as gridspec
import seaborn as sns, folium
from pathlib import Path
sns.set_theme(style='whitegrid',palette='husl')
plt.rcParams['figure.dpi'] = 120
Path('../data').mkdir(exist_ok=True)
print('Imports OK')

In [ ]:
df = pd.read_csv('../data/inserate_bereinigt.csv')
print(f'{len(df)} Inserate, {df["stadt"].nunique()} Staedte')

## 1. 4-Panel Dashboard

In [ ]:
fig = plt.figure(figsize=(16,12))
gs = gridspec.GridSpec(2,2,hspace=0.35,wspace=0.3)
order = df.groupby('stadt')['preis_chf'].median().sort_values(ascending=False).index

# Panel 1: Boxplot
ax1 = fig.add_subplot(gs[0,0])
sns.boxplot(data=df,y='stadt',x='preis_chf',order=order,palette='husl',ax=ax1)
ax1.set_title('Preisverteilung nach Stadt',fontweight='bold')
ax1.axvline(df['preis_chf'].median(),color='red',ls='--',alpha=0.7)

# Panel 2: Violinplot
ax2 = fig.add_subplot(gs[0,1])
zo = [z for z in ['1-1.5 Zi','2-2.5 Zi','3-3.5 Zi','4+ Zi'] if z in df['zimmer_gruppe'].values]
sns.violinplot(data=df,x='zimmer_gruppe',y='preis_chf',order=zo,palette='muted',ax=ax2,inner='quartile')
ax2.set_title('Preis nach Zimmeranzahl',fontweight='bold')

# Panel 3: Balken Preis/m2
ax3 = fig.add_subplot(gs[1,0])
pm2 = df.groupby('stadt')['preis_pro_m2'].mean().sort_values(ascending=False)
bars = ax3.barh(pm2.index,pm2.values,color=sns.color_palette('husl',len(pm2)))
ax3.set_title('Preis pro m2 nach Stadt',fontweight='bold')
for bar,val in zip(bars,pm2.values):
    ax3.text(val+0.3,bar.get_y()+bar.get_height()/2,f'CHF {val:.1f}',va='center',fontsize=9)

# Panel 4: Scatter
ax4 = fig.add_subplot(gs[1,1])
for s in order:
    sub = df[df['stadt']==s]
    ax4.scatter(sub['flaeche_m2'],sub['preis_chf'],alpha=0.5,s=25,label=s)
ax4.set_title('Flaeche vs. Preis',fontweight='bold')
ax4.legend(fontsize=8,ncol=2)

fig.suptitle('Schweizer Immobilienmarkt - Uebersicht',fontsize=15,fontweight='bold')
plt.savefig('../data/dashboard.png',dpi=150,bbox_inches='tight')
plt.show()
print('Gespeichert: dashboard.png')

## 2. Zusammenfassende Tabelle

In [ ]:
tab = df.groupby('stadt').agg(
    Anzahl=('preis_chf','count'), Preis=('preis_chf','mean'),
    Median=('preis_chf','median'), Flaeche=('flaeche_m2','mean'),
    Zimmer=('zimmer_anzahl','mean'), CHF_m2=('preis_pro_m2','mean')
).round(1).sort_values('Preis',ascending=False)
print('Zusammenfassende Tabelle:')
tab

## 3. Folium-Karte der Schweizer Staedte
Interaktive Karte mit Preisinfos pro Stadt.

In [ ]:
koordinaten = {
    'Zuerich':(47.3769,8.5417),'Genf':(46.2044,6.1432),
    'Bern':(46.9481,7.4474),'Basel':(47.5596,7.5886),
    'Luzern':(47.0502,8.3093),'Lausanne':(46.5197,6.6323),
    'Winterthur':(47.5003,8.7238),'St. Gallen':(47.4245,9.3767),
    'Lugano':(46.0037,8.9511),'Biel':(47.1368,7.2467),
}
stats = df.groupby('stadt')['preis_chf'].agg(['mean','count']).round(0)
karte = folium.Map(location=[46.8182,8.2275],zoom_start=8,tiles='CartoDB positron')
preise = [stats.loc[s,'mean'] for s in koordinaten if s in stats.index]
mi,ma = min(preise),max(preise)
def farbe(p):
    r = (p-mi)/(ma-mi) if ma>mi else 0.5
    return 'green' if r<0.33 else 'orange' if r<0.66 else 'red'
for s,(lat,lon) in koordinaten.items():
    if s not in stats.index: continue
    p = stats.loc[s,'mean']; n = int(stats.loc[s,'count'])
    folium.CircleMarker(
        location=[lat,lon], radius=12+n*0.05,
        color=farbe(p), fill=True, fill_color=farbe(p), fill_opacity=0.7,
        popup=folium.Popup(f'<b>{s}</b><br>CHF {p:,.0f}/Mt.<br>n={n}',max_width=150),
        tooltip=f'{s}: CHF {p:,.0f}'
    ).add_to(karte)
karte.save('../data/immobilien_karte.html')
print('Karte gespeichert: immobilien_karte.html')
print('Legende: gruen=guenstig, orange=mittel, rot=teuer')
karte